# Tool Calling — học theo notebook B4

**Mục tiêu:** chạy tool bằng tay trước, rồi hiểu chuyện gì xảy ra khi model đề nghị gọi tool. Ví dụ tra một thiết bị trong kho phòng học. Học theo thứ tự từ trên xuống. Phần chính dùng `langchain-core` và dữ liệu mẫu. Cell OpenAI ở cuối mặc định tắt.

Cách viết theo `B4_ToolCalling_langchain/notebooks/01_chung_tu_va_chi_phi.ipynb`: hàm Python → `@tool` → `.invoke()`.

## 1. Viết hàm Python trước

`rooms` là dữ liệu giả của backend. Hàm `find_room` nhận mã và trả phòng tương ứng. `rooms.get(asset_id)` trả `None` nếu mã vắng trong bảng.

In [ ]:
rooms = {"KIT-7": "B2", "KIT-8": "C1"}


def find_room(asset_id: str) -> str | None:
    return rooms.get(asset_id)


print(find_room("KIT-7"))  # B2
print(find_room("KIT-0"))  # None

## 2. Bọc hàm bằng `@tool` rồi gọi bằng tay

`@tool` giúp LangChain lấy tên hàm, mô tả trong docstring và tham số từ khai báo kiểu. Tool dưới dùng lại `find_room`.

`lookup_room.invoke({"asset_id": "KIT-7"})` chạy tool ngay trong Python. Tên khóa `asset_id` khớp tên tham số trong hàm.

In [ ]:
from langchain_core.tools import tool


@tool
def lookup_room(asset_id: str) -> dict:
    """Tra phòng của một thiết bị theo mã thiết bị."""
    room = find_room(asset_id)
    if room is None:
        return {"error": "Không tìm thấy thiết bị"}
    return {"asset_id": asset_id, "room": room}


result = lookup_room.invoke({"asset_id": "KIT-7"})
print(result)
assert result["room"] == "B2"
assert lookup_room.invoke({"asset_id": "KIT-0"}) == {"error": "Không tìm thấy thiết bị"}

## 3. Dòng `bind_tools(...).invoke(...)` làm gì?

Mẫu trong B4 dùng hai thao tác nối nhau:

```python
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chosen_tool = lookup_room
first = llm.bind_tools([chosen_tool], tool_choice=chosen_tool.name).invoke(
    [HumanMessage(content=prompt)]
)
```

| Đoạn | Điều xảy ra |
|---|---|
| `ChatOpenAI(...)` | Tạo đối tượng để gọi model. |
| `llm.bind_tools([chosen_tool], ...)` | Gắn khai báo tool gồm tên, mô tả và tham số. `tool_choice=chosen_tool.name` yêu cầu model chọn đúng tool này trong ví dụ một tool. |
| `.invoke([HumanMessage(content=prompt)])` | Gửi câu hỏi cùng khai báo tool lên OpenAI. Model đọc chúng và trả phản hồi `first`. |
| `first.tool_calls[0]` | Lấy đề nghị gọi đầu tiên. Đề nghị chứa `name`, `args` và `id`. Ví dụ `args` là `{"asset_id": "KIT-7"}`. |

**Dừng ở đây thì mới có đề nghị.** Hàm Python thực sự chạy ở dòng `chosen_tool.invoke(call["args"])` trong bước sau. Bạn chưa cần đọc source nội bộ LangChain để theo dõi luồng này.

## 4. Thực thi đề nghị, rồi chuẩn bị trả kết quả cho model

Dưới đây là một `call` mẫu để tập offline. Backend kiểm tên có đúng tool được phép hay không rồi chạy tool. `ToolMessage` chứa kết quả dưới dạng JSON. `tool_call_id` lấy từ `call["id"]` để model biết kết quả này thuộc lời gọi nào.

In [ ]:
import json
from langchain_core.messages import ToolMessage

chosen_tool = lookup_room
call = {
    "name": "lookup_room",
    "args": {"asset_id": "KIT-7"},
    "id": "call_1",
}

if call["name"] != chosen_tool.name:
    raise ValueError("Tool không được phép")

result = chosen_tool.invoke(call["args"])
tool_reply = ToolMessage(
    content=json.dumps(result, ensure_ascii=False),
    tool_call_id=call["id"],
)

print(tool_reply.content)
print(tool_reply.tool_call_id)
assert tool_reply.tool_call_id == "call_1"

## 5. Tự thử với OpenAI khi sẵn sàng

Cell này gom đúng luồng của B4: gọi model lần đầu để lấy đề nghị → chạy tool → gọi model lần hai để viết câu trả lời. Lần hai nhận câu hỏi, phản hồi `first` có yêu cầu gọi tool và `ToolMessage` chứa kết quả.

Mặc định tắt. Khi tự bật, cần `langchain-openai`, `OPENAI_API_KEY` trong môi trường và quyền dùng model được chọn. Tên model lấy từ notebook B4.

In [ ]:
RUN_LLM_DEMO = False

if RUN_LLM_DEMO:
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI

    prompt = "Thiết bị KIT-7 ở phòng nào?"
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    first = llm.bind_tools(
        [chosen_tool], tool_choice=chosen_tool.name
    ).invoke([HumanMessage(content=prompt)])

    print("Đề nghị:", first.tool_calls)

    if first.tool_calls:
        call = first.tool_calls[0]
        if call["name"] != chosen_tool.name:
            raise ValueError("Tool không được phép")

        result = chosen_tool.invoke(call["args"])
        print("Kết quả Python:", result)
        tool_reply = ToolMessage(
            content=json.dumps(result, ensure_ascii=False),
            tool_call_id=call["id"],
        )
        final = llm.invoke([HumanMessage(content=prompt), first, tool_reply])
        print("Trả lời:", final.content)
    else:
        print("Model chưa đề nghị gọi tool.")

**Tự thử:** ở bước 4, đổi mã thành `KIT-8`. Trước khi chạy, đoán nội dung `ToolMessage`. Tiếp theo, đổi `name` thành tên khác và xem dòng nào dừng chương trình.

Khi làm BT2, thay từ điển mẫu bằng dữ liệu backend. Học từng lớp khi cần: hàm thuật toán Python → tool → SQLite/SQLAlchemy → FastAPI và test → web chat. Mở [lộ trình stack B4–B7](../TOOL_CALLING_GUIDE.md) và [contract từng bài](../EXERCISE_CONTRACTS.md) khi đến lớp tương ứng.